In [5]:
import numpy as np
from tqdm import tqdm

In [6]:
p_win = 0.85

house_edge = 0.01
balance = 100.0
multiplier = np.round((1 - house_edge) / p_win, 4)
init_bet_size = 0.15
num_rolls = balance / init_bet_size

print(f"Multiplier: {multiplier} with probability of winning: {p_win}")
print(f'Currenct balance: {balance}')

Multiplier: 1.1647 with probability of winning: 0.85
Currenct balance: 100.0


In [7]:
E_num_win = num_rolls * p_win
E_winnings = (E_num_win * init_bet_size * multiplier) - (num_rolls * init_bet_size)
print(f'Expected winnings: {E_winnings} with {E_num_win} wins')

Expected winnings: -1.0004999999999882 with 566.6666666666667 wins


# Strategy 

In [76]:
def dice_sim(budget, upper_bound, lower_bound, init_bet_size, p_win, games_total, bet_alpha=1.4):
    current_balance = budget
    bet_size = init_bet_size
    multiplier = (1- 0.00) / p_win
    over = True

    games_won, profit = 0, 0.0
    games_lost = 0
    
    for i in range(games_total):
        while True:  # Keep playing until we reach the upper or lower bound
            rand = np.random.rand()
            # print(f'------ p_win = {p_win}; rand = {rand} ------')
            if over:
                if rand > p_win:
                    current_balance += bet_size * multiplier
                    bet_size = init_bet_size
                    over = False if over else True
                    # print(f'Current balance: {current_balance}, bet size: {bet_size}')
                else:
                    current_balance -= bet_size
                    bet_size *= bet_alpha
                    # print(f'Current balance: {current_balance}, bet size: {bet_size}')
                    
            else:
                if rand < p_win:
                    current_balance += bet_size * multiplier
                    bet_size = init_bet_size
                    over = False if over else True
                    # print(f'Current balance: {current_balance}, bet size: {bet_size}')
                else:
                    current_balance -= bet_size
                    bet_size *= bet_alpha
                    # print(f'Current balance: {current_balance}, bet size: {bet_size}')
                    
            if current_balance > upper_bound:
                games_won += 1
                profit += (current_balance - budget)
                current_balance = budget
                over = True
                bet_size = init_bet_size
                break
            elif current_balance < lower_bound:
                games_lost += 1
                current_balance = budget
                over = True
                bet_size = init_bet_size
                break
    
    overal_profit = profit - (games_lost * (budget - lower_bound))
    return games_won/games_total, games_lost/games_total, overal_profit

In [77]:
num_sessions = 100_000
games_per_session = 5
profits, wins, losses = [], [], []

budget = 200.0
upper_bound = 400.0
lower_bound = 100.0
init_bet_size = 0.15
bet_alpha = 1.8

for _ in tqdm(range(num_sessions)):
    games_won, games_lost, profit = dice_sim(budget=budget, upper_bound=upper_bound, lower_bound=lower_bound, init_bet_size=init_bet_size, p_win=0.1076, games_total=games_per_session, bet_alpha=bet_alpha)
    
    profits.append(profit)
    wins.append(games_won)
    losses.append(games_lost)
    
print(f'Average profit: {np.mean(profits)} with (sigma: {np.std(profits)})')
print(f'Average wins: {np.mean(wins)} with (sigma: {np.std(wins)})')
print(f'Average losses: {np.mean(losses)} with (sigma: {np.std(losses)})')

print("-----------------------------")

print(f'Profit (lowest estimate): {np.mean(profits) - 2 * np.std(profits)}')
print(f'Profit (highest estimate): {np.mean(profits) + 2 * np.std(profits)}')



100%|██████████| 100000/100000 [00:05<00:00, 17107.23it/s]


Average profit: 244.9713224404335 with (sigma: 622.5409068206425)
Average wins: 0.28154400000000007 with (sigma: 0.20157027574521)
Average losses: 0.7184560000000001 with (sigma: 0.20157027574521)
-----------------------------
Profit (lowest estimate): -1000.1104912008515
Profit (highest estimate): 1490.0531360817185
